## Accepted historical trees summary replay

Read v2 summaries and date bases, then draw one new display chart from those historical values. No model fits, retraining or Chapter 39 row reconstruction occur. Real-data research remains blocked; the separate fresh CLI is not invoked here.

[Entry modes and preparation](../../../NOTEBOOK_GUIDE.md). This stage delivers language views; portable dependencies and execution are checked in the later isolated cold start.

# Project 6 v2: trees, stability and ablation

The historical v2 configuration inherited v1 labels, folds, inner validation, portfolio rules and ledger, adding a tree, random forest, HGB and PCA+Ridge ablation. HGB internal early stopping was disabled and rounds selected in inner date folds. The configuration was frozen after seeing v1 constructed results, not before a real-market experiment.

Signal closes at $D_k$, trade at $D_{k+1}$ and label maturity at $D_{k+6}$. Teaching sources have been accepted; real research and formal final verification remain incomplete.

In [ ]:
from pathlib import Path
import os, sys, uuid
ROOT = next(p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir())
for path in (ROOT / "src", ROOT / "code", ROOT):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
os.environ["MPLBACKEND"] = "Agg"
os.chdir(ROOT)
RUN_DIR = ROOT / "runs" / ("p06_trees_v2_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
print("Run directory:", RUN_DIR.relative_to(ROOT))
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
P06 = ROOT / "fixtures/projects/project06"
print("P6 trees: historical accepted CONSTRUCTED result display; no fitting/retraining; real route blocked")


## 1. Small fitting cell: skipped

The original reduced eight-asset/420-day fitting cell is disabled. Its old “live run, about 20 seconds” description does not apply. No fresh training result is produced.

In [ ]:
print("Original small model fitting cell SKIPPED in I2: no P6 retraining and no historical Ch39 repair")


## 2. Accepted historical summaries and date bases

Read v2_ic_date_bases.json and base/regime summaries from shared fixtures. Own IC averages use each model's valid dates; delta IC uses its intersection with Ridge. Sparse models can have constant-prediction dates. Missing-date series have NaN t-display because compressed NW(4) is not trading-day HAC. Baseline equality checks refer to the historical A8-A c3 identity. Save one new historical_constructed_trees_sharpe.png, a display plot rather than a retrained result.

In [ ]:
# IC dates: own IC dates differ for sparse models (constant predictions on some dates). t-values are shown only for
# series with IC on every mature-label date; for the others the frozen summary's t was computed on a compressed
# series of valid dates and is not a trading-day NW(4) statistic (rework R01).
bases = json.loads((P06 / "v2_ic_date_bases.json").read_text())["scenarios"]
rows = []
for scen in ("base", "regime"):
    s = json.loads((P06 / f"v2_summary_{scen}.json").read_text())
    assert s["data_class"] == "CONSTRUCTED" and s["baseline_check"]["ok"]
    for m, r in s["metrics"].items():
        if m in ("zero", "train_mean"):
            continue
        d, b = r.get("delta_ic_vs_ridge", {}), bases[scen][m]
        assert b["own_mean_matches_frozen"] and b.get("delta_matches_frozen", True)
        contiguous = b["frozen_t_is_trading_day_hac"]
        rows.append({"scenario": scen, "model": m, "R2_vs_mean_%": 100 * r["prediction"]["oos_r2_vs_train_mean"],
                     "IC": r["prediction"]["ic_mean"], "IC_dates": b["own_ic_dates"],
                     "IC_t": r["prediction"]["ic_t"] if contiguous else np.nan,
                     "dIC_vs_ridge": d.get("mean"), "paired_dates": b.get("paired_dates"),
                     "ridge_on_paired": b.get("ridge_paired_mean"), "dIC_t": d.get("t_nw") if contiguous else np.nan,
                     "net10_SR": r["portfolio"]["0.001"]["sharpe"], "net20_SR": r["portfolio"]["0.002"]["sharpe"],
                     "turnover": r["portfolio"]["0.001"]["turnover_ann"]})
full = pd.DataFrame(rows).set_index(["scenario", "model"])
display(full.round(4))
print("NaN t = not reported: the series has dates without a defined IC (see v2_ic_date_bases.json).")
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6), sharey=True)
for a, scen in zip(ax, ("base", "regime")):
    t = full.loc[scen]
    a.bar(t.index, t["net10_SR"], color=["#247F82" if m in ("tree", "rf", "gbm") else "#7F7F7F" for m in t.index])
    a.axhline(0, color="#444444", lw=0.6); a.set_title(f"{scen}: Sharpe after 10 bp (CONSTRUCTED)"); a.tick_params(axis="x", rotation=60)
plt.tight_layout(); fig.savefig(RUN_DIR / "historical_constructed_trees_sharpe.png",dpi=130); plt.show()

## 3. Historical ablation, importance stability and seeds

Display frozen ablation IC, delta IC, costed Sharpe, consecutive importance-rank correlations and seed summaries. These are archived results, not fresh sweeps.

In [ ]:
s = json.loads((P06 / "v2_summary_base.json").read_text())
abl = pd.DataFrame({k: {"IC": v["prediction"]["ic_mean"], "dIC_vs_full": v["delta_ic_vs_full"]["mean"],
                        "net10_SR": v["portfolio"]["0.001"]["sharpe"]} for k, v in s["ablation"].items()}).T
display(abl.round(4))
print({m: round(v["consecutive_rank_corr_mean"], 3) for m, v in s["importance_stability"].items()})
print({m: {sd: round(r["portfolio"]["0.001"]["sharpe"], 3) for sd, r in v.items() if sd != "prediction_corr"} for m, v in s["seeds"].items()})